# M3L2 - Sourcing Data For Analysis Pipelines

This Notebook provides an overview of the SRA toolkit, and we will use SRA tools to download our files for RNA-Seq pseudo-alignment using salmon. 

**NOTE: The bash kernel and the global environment `class/mendoza/class-bbme4001-7001/v1.0` should be loaded within this Notebook. Please refer to Module 2 Practice 2 for instructions on how to add the bash kernel to your notebook.**

**By the end of this notebook, you should be familiar with the following concepts:** 
- Be familiar with navigating the GEO accession website
- Converting GSM to SRA IDs
- Understand the OPT3 dataset we will use for this week's analysis
- Download and relabel all the `fastq` files according to their sample name within the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series within your `m3_project_pseudo_alignment` folder. 

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

In [ ]:
#load the global environment from Module 2
ml class/mendoza/class-bbme4001-7001/v1.0
#This command makes the terminal known within the bash shell
export TERM=xterm

#change directory to the project folder
cd ..
cd m3_project_pseudo_alignment
#current working directory
pwd



## What is the SRA Toolkit?

The SRA Toolkit is NCBI’s command-line collection for accessing and converting data stored in the Sequence Read Archive ([SRA](https://www.ncbi.nlm.nih.gov/sra)). It includes utilities to download SRA objects (`prefetch`), extract FASTQ (`fasterq-dump`), inspect SRA files and configure behavior (`vdb-config`), and more. `fasterq-dump` is the faster successor to the older `fastq-dump`.

From this toolkit, we can quickly download a list of sample files from the SRA database to use them within our RNA-Seq pipelines. To use SRA or the GEO accession efficiently, it is important to understand how the data is stored. We will use the example of our experiment stored in the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series.

![image.png](images/M3L2_Sourcing_Data_For_Analysis_Pipelines_0e5ef6a0.png)

The above image shows the home page for the experiment "Transcript profiling of the roots and leaves of opt3-2 mutant and wild type Arabidopsis" stored in the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series. This experiment is part of the SuperSeries [GSE128157](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE128157). Please visit the GSE series in this example, and practice navigating the page. 

-----

## How are GSE's Organized? 
Each GSE series contains an overall experiment, either as a series (one experiment), a SuperSeries (a collection of experiments), or a SubSeries (an experiment that is part of a SuperSeries). Our example, [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) contains the sample **GSM's** for each sample within the experiments GSE series. Each GSE sample represents a library from a genotype/treatment/replicate condition from within the study. Each of these GSM samples links to one or more experiments SRX. An SRX is an experiment describing sequencing conditions within the SRA database. Each SRX experiment contains one or more runs of the samples within it, which are denoted as SRR's. These SRR's are the raw files you will retrieve using the `SRA ToolKit`'s `prefetch` command. See the table below for a description of how our experiment([GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275), which compares *A. thaliana* OPT-3 mutants to the *A. thaliana* Col-0 control in replete conditions) is stored within the GSE and SRA databases on NCIB:
| Accession Type       | Format Example                 | What It Represents                                                                               | Example in GSE79275                                                                                                                 |
| -------------------- | ------------------------------ | ------------------------------------------------------------------------------------------------ | ----------------------------------------------------------------------------------------------------------------------------------- |
| **Series (GSE)**     | `GSE79275`                     | A study/series: a grouped set of samples, often with a common goal or design.                    | The entire experiment “Transcript profiling … roots and leaves of opt3-2 mutant and wild type Arabidopsis” in GSE79275. ([NCBI][1]) |
| **Sample (GSM)**     | `GSM2090459` (and many others) | An individual sample (biological replicate, library, etc) within a series.                       | In GSE79275 you see many GSM IDs (e.g., GSM2090459, GSM2090460 … up to GSM2090494). ([NCBI][1])                                     |
| **Experiment (SRX)** | e.g., `SRX…`                   | An SRA Experiment record: describes the sequencing experiment (library, conditions) tied to SRA. | GSE79275 lists its SRA relation as `SRP071838` (! “SRA Run Selector” link) – under that you will have SRX IDs. ([NCBI][1])          |
| **Run (SRR)**        | e.g., `SRR…`                   | The actual sequencing run(s): what you download (FASTQ) and work with downstream.                | Within the SRA link for this study you will find SRR IDs (one per run) associated with each sample.                                 |

[1]: https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275 "
    GEO Accession viewer
    "


## How do we map out GSM samples to the SRA database to find their SRR sample accession?

For us to download our the raw files from our GSE experiment, we will need to use the SRR accession so that we can query the SRA database. We will use the `Entrez Direct` package within the bash command line. We can use the `entrez` tool to gather our run info for each of our GSM's within the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series by searching the `gds` database.
**Note: This works within our `bash` kernel in Jupyter notebook, but these commands will run fine as well within our terminal.**

In [ ]:
esearch -db gds -query "GSE79275" | efetch -format runinfo > references/SraRunTable.txt

### For our experiment, we will want to use the following samples to compare the effects of the OPT-3 mutation with our Col-0 Controls: 

| Sample ID | GEO Accession | Condition | Replicate | Description |
|------------|----------------|------------|------------|--------------|
| **CRN1-1** | GSM2090459 | C (Col-0 untreated) | N-1 | Col-0 untreated root sample |
| **CRN2-1** | GSM2090462 | C (Col-0 untreated) | N-2 | Col-0 untreated root sample |
| **CRN3-1** | GSM2090465 | C (Col-0 untreated) | N-3 | Col-0 untreated root sample |
| **ORN1-1** | GSM2090477 | O (OPT3-untreated)| N-1 | OPT3 untreated root sample |
| **ORN2-1** | GSM2090480 | O (OPT3-untreated) | N-2 | OPT3 untreated root sample |
| **ORN3-1** | GSM2090483 | O (OPT3-untreated) | N-3 | OPT3 untreated root sample |

We know that we only want the 6 samples within our SRR query, so let's parse the `SraRunTable.txt` file with awk to create a csv table that includes our sample name, GSM, and SRX accessions. We can then use the SRX accession to find the SRR accession and download the `fastq.gz` files for each of our samples. There are other ways to do this, but this is a good demonstration of the power of `AWK`. See if you can understand the AWK command. 

In [ ]:
#Set the 6 samples we want
SAMPLES="CRN1-1 CRN2-1 CRN3-1 ORN1-1 ORN2-1 ORN3-1"

#Parse the SraRunTable.txt file for the sample name, GSE, and SRX accessions with AWK
awk -v samples="$SAMPLES" '
/Sample[[:space:]]+Accession:/ { gsm=$3 }
/SRA Run Selector:/ { 
    match($0,/SRX[0-9]+/,srx)
    if(name_in_list) print name_in_list "," gsm "," srx[0]
    name_in_list=""
}
/^[0-9]+\. / {
    match($0, /[A-Z]+[0-9]+-[0-9]+/, n)
    split(samples,a," ")
    for(i in a) if(a[i]==n[0]) { name_in_list=n[0]; break }
}
' references/SraRunTable.txt > references/selected_samples.csv

We now have a table that contains the following: Treatment name, GSE, and SRX:

![image.png](images/M3L2_Sourcing_Data_For_Analysis_Pipelines_45f4fc37.png)


Run the code cell below to fetch each of the `fastq.gz` files for our 6 samples using `fasterq-dump` to access the SRR's found by the esearch query to the SRA data base.

In [ ]:
mkdir -p data/raw
CSV="references/selected_samples.csv"

# Open the CSV on fd 3
exec 3< "$CSV"

echo "passes check for fd3"
while IFS=',' read -r sample gsm srx <&3; do
    # Check if R1 already exists
    if [[ -f "data/raw/${sample}_R1.fastq.gz" ]]; then
        echo "Skipping $sample ($srx), files already exist."
        continue
    fi

    echo "Processing $sample ($srx)..."

    # Get SRRs
    mapfile -t srrs < <(esearch -db sra -query "$srx" | efetch -format runinfo | tail -n +2 | cut -d, -f1)

    for srr in "${srrs[@]}"; do
        echo "Downloading $srr for $sample..."
        fasterq-dump "$srr" --outdir data/raw --split-files

        # Compress and rename
        for fq in data/raw/${srr}*.fastq; do
            [[ -f "$fq" ]] || continue
            gzip -f "$fq"
            base=$(basename "$fq" .fastq)
            if [[ $base == *_1 ]]; then
                mv "data/raw/${base}.fastq.gz" "data/raw/${sample}_R1.fastq.gz"
            elif [[ $base == *_2 ]]; then
                mv "data/raw/${base}.fastq.gz" "data/raw/${sample}_R2.fastq.gz"
            else
                mv "data/raw/${base}.fastq.gz" "data/raw/${sample}_R1.fastq.gz"
            fi
        done
    done

done

# Close fd 3
exec 3<&-

With that we have downloaded all of the `fastq` files we need for our pseduo alignment from the GSE accession. They should be stored according to their sample name from the [GSE79275](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE79275) series within your `m3_project_pseudo_alignment` folder. 

In [ ]:
#check the layout of your m3_project folder:
tree